# Yelp Fake Review Detection — sklearn Models (8 classifiers)
**Training data:** YelpZip + YelpChi + YelpNYC — balanced to ~252k reviews (all fake + equal genuine sample)

**Labels:** -1 / Y = fake (1) &nbsp;|&nbsp; 1 / N = genuine (0)

**Output:** 8 joblib pipeline files (`yelp_fake_*.joblib`)

---
### Instructions
1. Run `prepare_yelp_split.ipynb` first — it generates `yelp_split.csv` on your Google Drive.
2. Update the path in **cell 3** if your `yelp_split.csv` is in a subfolder.
3. Run all cells in order.
4. Download `yelp_fake_sklearn_models.zip` at the end and extract all `.joblib` files into your `data/` folder.

In [ ]:
!pip install -q xgboost joblib scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

# ── UPDATE THIS PATH if your split file is in a different location ──
SPLIT_CSV  = '/content/drive/MyDrive/yelp_split.csv'
OUTPUT_DIR = '/content/yelp_models'

# Remove stale output from any previous run so old files don't end up in the zip
if os.path.isdir(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
    print(f'Removed old output folder: {OUTPUT_DIR}')
os.makedirs(OUTPUT_DIR)
print(f'Output folder ready: {OUTPUT_DIR}')

In [ ]:
import random
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from xgboost import XGBClassifier

SEED       = 42
KNN_SAMPLE = 20_000
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
df_split = pd.read_csv(SPLIT_CSV)

X_tr  = df_split[df_split['split'] == 'train']['text'].tolist()
y_tr  = df_split[df_split['split'] == 'train']['label'].tolist()
X_val = df_split[df_split['split'] == 'val']['text'].tolist()
y_val = df_split[df_split['split'] == 'val']['label'].tolist()
X_te  = df_split[df_split['split'] == 'test']['text'].tolist()
y_te  = df_split[df_split['split'] == 'test']['label'].tolist()

print(f'Train: {len(X_tr):,}   Val: {len(X_val):,}   Test: {len(X_te):,}')

print('\n=== Split sanity check ===')
print(df_split['split'].value_counts())
print()
print(pd.crosstab(df_split['split'], df_split['label'], normalize='index').round(3))
print()
print('Missing values:')
print(df_split[['text', 'label', 'split']].isna().sum())
print('Label values:', sorted(df_split['label'].unique()))

In [ ]:
# Balanced sample for KNN — drawn from training set only
# KNN does not scale to the full 200k train set, so it trains on a fixed 20k subsample
tr_fake_idx    = [i for i, l in enumerate(y_tr) if l == 1]
tr_genuine_idx = [i for i, l in enumerate(y_tr) if l == 0]
rng = random.Random(SEED)
knn_idx = rng.sample(tr_fake_idx, KNN_SAMPLE // 2) + rng.sample(tr_genuine_idx, KNN_SAMPLE // 2)
rng.shuffle(knn_idx)
knn_texts  = [X_tr[i] for i in knn_idx]
knn_labels = [y_tr[i] for i in knn_idx]

print(f'KNN training subset: {len(knn_texts):,}')

In [ ]:
def make_pipeline(classifier):
    return Pipeline([
        ('tfidf', TfidfVectorizer(max_features=50_000, ngram_range=(1, 2),
                                  stop_words='english', sublinear_tf=True)),
        ('clf', classifier),
    ])

def print_metrics(y_true, y_pred):
    print(f'  Accuracy  : {accuracy_score(y_true, y_pred):.1%}')
    print(f'  Macro F1  : {f1_score(y_true, y_pred, average="macro", zero_division=0):.1%}')
    print(f'  Fake  P/R : {precision_score(y_true, y_pred, zero_division=0):.1%} / {recall_score(y_true, y_pred, zero_division=0):.1%}')

model_defs = [
    # (name, pipeline, use_knn_sample)
    # KNN: trained and retrained on its 20k subsample only (full train+val is too slow for KNN)
    ('knn',      make_pipeline(KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),                         True),
    ('rf',       make_pipeline(RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED)), False),
    ('dt',       make_pipeline(DecisionTreeClassifier(random_state=SEED)),                              False),
    ('lr',       make_pipeline(LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1)),        False),
    ('nb',       make_pipeline(MultinomialNB()),                                                        False),
    ('xgb',      make_pipeline(XGBClassifier(n_estimators=100, use_label_encoder=False,
                                              eval_metric='logloss', random_state=SEED, n_jobs=-1)),    False),
    ('adaboost', make_pipeline(AdaBoostClassifier(n_estimators=100, random_state=SEED)),                False),
    # MLP: max_iter=50 is a lightweight baseline — convergence warnings suppressed intentionally
    ('mlp',      make_pipeline(MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=50,
                                             random_state=SEED)),                                       False),
]

print(f'{len(model_defs)} classifiers defined.')

In [ ]:
import json

W = 56
print('=' * W)
print('YELP FAKE DETECTION — Training 8 classifiers')
print('=' * W)

thresholds   = {}
metrics_rows = []

for name, pipeline, use_knn_sample in model_defs:
    print(f'\n[{name.upper()}] Training ...')

    Xtr = knn_texts  if use_knn_sample else X_tr
    ytr = knn_labels if use_knn_sample else y_tr

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        pipeline.fit(Xtr, ytr)

    # Threshold tuning on validation set — optimise macro F1, never touch test set
    val_probs = pipeline.predict_proba(X_val)[:, 1]
    best_thresh, best_val_f1 = 0.5, 0.0
    for t in np.arange(0.10, 0.90, 0.01):
        f = f1_score(y_val, (val_probs >= t).astype(int), average='macro', zero_division=0)
        if f > best_val_f1:
            best_val_f1 = f
            best_thresh = round(float(t), 2)

    # Evaluate on test set using best threshold
    test_probs = pipeline.predict_proba(X_te)[:, 1]
    y_pred = (test_probs >= best_thresh).astype(int)
    print(f'  Best threshold: {best_thresh:.2f}  (Val macro F1: {best_val_f1:.1%})')
    print_metrics(y_te, y_pred)

    thresholds[name] = best_thresh
    metrics_rows.append({
        'model':      name,
        'threshold':  best_thresh,
        'accuracy':   round(accuracy_score(y_te, y_pred), 4),
        'macro_f1':   round(f1_score(y_te, y_pred, average='macro', zero_division=0), 4),
        'precision':  round(precision_score(y_te, y_pred, zero_division=0), 4),
        'recall':     round(recall_score(y_te, y_pred, zero_division=0), 4),
        'n_test':     len(y_te),
        'n_genuine':  sum(1 for y in y_te if y == 0),
        'n_fake':     sum(1 for y in y_te if y == 1),
    })

    # Save the exact model that was evaluated — threshold is calibrated to this model
    path = os.path.join(OUTPUT_DIR, f'yelp_fake_{name}.joblib')
    joblib.dump(pipeline, path)
    print(f'  Saved → {path}')

# Save thresholds JSON — serving code must load this and apply thresholds manually
thresh_path = os.path.join(OUTPUT_DIR, 'yelp_fake_thresholds.json')
with open(thresh_path, 'w') as f:
    json.dump(thresholds, f, indent=2)
print(f'\nThresholds saved → {thresh_path}')

# Save metrics CSV for thesis reporting
metrics_path = os.path.join(OUTPUT_DIR, 'yelp_fake_sklearn_metrics.csv')
pd.DataFrame(metrics_rows).to_csv(metrics_path, index=False)
print(f'Metrics saved    → {metrics_path}')

print(f'\n{"=" * W}')
print('Done. All models saved.')
print('=' * W)

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive('/content/yelp_fake_sklearn_models', 'zip', OUTPUT_DIR)
print(f'Archive: {zip_path}')
files.download(zip_path)

## Next steps
1. Extract `yelp_fake_sklearn_models.zip`
2. Place all files in your `data/` folder:
```
data/yelp_fake_knn.joblib
data/yelp_fake_rf.joblib
data/yelp_fake_dt.joblib
data/yelp_fake_lr.joblib
data/yelp_fake_nb.joblib
data/yelp_fake_xgb.joblib
data/yelp_fake_adaboost.joblib
data/yelp_fake_mlp.joblib
data/yelp_fake_thresholds.json   ← apply these thresholds in serving code
data/yelp_fake_sklearn_metrics.csv
```
The serving code must load `yelp_fake_thresholds.json` and use `predict_proba()[:, 1] >= threshold` instead of `predict()`.